Loading metdata and normalised csv files from milestone 1.

One more thing. 
Upon looking at the dataset again, found out that not only do we have mutliple samples and mutiple input features but also mutiple output labels to predict too.
So now, instead of breaking the input features into batch sizes and then training per ONE label (as we thought there was one output label only),
we will now do batch training by using all input features for each output label class per each classifier.
We will now attempt to train many per-label classifiers in parallel batches across nodes.
This avoids needing one node per output label while keeping Binary Relevance correct.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

np.random.seed(42)

TRAIN_FILE = Path("eurlex_normalized.csv")
TEST_FILE = Path("eurlex_test_normalized.csv")
NORM_FILE = Path("normalization_params.pkl")
PARTITION_META_FILE = Path("partition_metadata.pkl")

if not TRAIN_FILE.exists() or not TEST_FILE.exists():
    raise FileNotFoundError("Required normalized CSV files from milestone 1 are missing.")

df_train = pd.read_csv(TRAIN_FILE)
df_test = pd.read_csv(TEST_FILE)

norm_params = None
partition_meta = None
if NORM_FILE.exists():
    with open(NORM_FILE, "rb") as f:
        norm_params = pickle.load(f)
if PARTITION_META_FILE.exists():
    with open(PARTITION_META_FILE, "rb") as f:
        partition_meta = pickle.load(f)

feature_cols = [c for c in df_train.columns if c.startswith("f")]

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")
print(f"Number of features: {len(feature_cols)}")
print(f"Normalization params loaded: {norm_params is not None}")
print(f"Partition metadata loaded: {partition_meta is not None}")

Train shape: (15377, 5002)
Test shape: (3971, 5001)
Number of features: 5000
Normalization params loaded: True
Partition metadata loaded: True


Preparing the feature matrix and converting multi-label strings into binary targets for selected labels.
The selected labels are used for small-scale classifier validation.

In [3]:
def parse_labels(label_str):
    if pd.isna(label_str):
        return []
    s = str(label_str).strip()
    if s == "":
        return []
    labels = []
    for item in s.split():
        parts = item.split(":")
        try:
            labels.append(int(parts[0]))
        except ValueError:
            continue
    return labels

X = df_train[feature_cols].values
all_label_lists = df_train["labels"].apply(parse_labels)

label_freq = {}
for row_labels in all_label_lists:
    for lbl in row_labels:
        label_freq[lbl] = label_freq.get(lbl, 0) + 1

# Small-scale validation set of labels for Mustafa's task
TOP_K_LABELS = 12
selected_labels = sorted(label_freq, key=label_freq.get, reverse=True)[:TOP_K_LABELS]

print(f"Parsed labels for {len(all_label_lists)} training rows")
print(f"Unique labels found: {len(label_freq)}")
print(f"Selected labels for small-scale validation: {selected_labels[:6]} ...")

Y_small = pd.DataFrame(index=df_train.index)
for lbl in selected_labels:
    Y_small[f"label_{lbl}"] = all_label_lists.apply(lambda lst: int(lbl in lst))

print(f"Feature matrix shape: {X.shape}")
print(f"Small-scale target matrix shape: {Y_small.shape}")

Parsed labels for 15377 training rows
Unique labels found: 3806
Selected labels for small-scale validation: [3902, 2640, 1996, 3303, 1367, 1848] ...
Feature matrix shape: (15377, 5000)
Small-scale target matrix shape: (15377, 12)


Defining the Binary Classifier as a class so it can be reused.

In [4]:
class BinaryClassifier:
    def __init__(self, C=1.0, max_iter=1000, class_weight="balanced", random_state=42):
        self.C = C
        self.max_iter = max_iter
        self.class_weight = class_weight
        self.random_state = random_state
        self.model = LogisticRegression(
            C=self.C,
            max_iter=self.max_iter,
            class_weight=self.class_weight,
            solver="liblinear",
            random_state=self.random_state,
        )

    def fit(self, X_train, y_train):
        self.model.fit(X_train, y_train)
        return self

    def predict(self, X_eval):
        return self.model.predict(X_eval)

    def predict_proba(self, X_eval):
        if hasattr(self.model, "predict_proba"):
            return self.model.predict_proba(X_eval)[:, 1]
        return None

print("BinaryClassifier class ready")

BinaryClassifier class ready


This step trains one binary model per selected label using class balancing and stores per-label models for evaluation.

In [5]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X,
    Y_small,
    test_size=0.2,
    random_state=42,
)

label_models = {}
for col in Y_train.columns:
    y_col = Y_train[col].values

    # Skip labels with single-class training data
    if len(np.unique(y_col)) < 2:
        continue

    clf = BinaryClassifier(C=1.0, max_iter=1000, class_weight="balanced", random_state=42)
    clf.fit(X_train, y_col)
    label_models[col] = clf

print(f"Validation split: X_train={X_train.shape}, X_val={X_val.shape}")
print(f"Models trained: {len(label_models)} / {Y_train.shape[1]}")

Validation split: X_train=(12301, 5000), X_val=(3076, 5000)
Models trained: 12 / 12


This step runs small-scale validation and reports per-label and average metrics for baseline implementation.

In [7]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

rows = []
for col, clf in label_models.items():
    y_true = Y_val[col].values
    y_pred = clf.predict(X_val)
    y_prob = clf.predict_proba(X_val)

    report = {
        "label": col,
        "support_pos": int(y_true.sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }

    if y_prob is not None and len(np.unique(y_true)) > 1:
        report["log_loss"] = float(log_loss(y_true, y_prob, labels=[0, 1]))
    else:
        report["log_loss"] = np.nan

    rows.append(report)

metrics_df = pd.DataFrame(rows).sort_values("f1", ascending=False)

if not metrics_df.empty:
    print("Per-label metrics (top rows):")
    display(metrics_df.head(10))

    avg_metrics = metrics_df[["accuracy", "precision", "recall", "f1", "log_loss"]].mean(numeric_only=True)
    print("Average validation metrics across trained labels:")
    print(avg_metrics)
else:
    print("No valid label models were trained. Try increasing TOP_K_LABELS or checking label distribution.")

Per-label metrics (top rows):


,label,support_pos,accuracy,precision,recall,f1,log_loss
2,label_1996,163,0.989272,0.877907,0.926380,0.901493,0.119194
6,label_882,120,0.960663,0.497696,0.900000,0.640950,0.218061
0,label_3902,199,0.871586,0.320513,0.879397,0.469799,0.340328
5,label_1848,140,0.913524,0.325000,0.835714,0.468000,0.296998
3,label_3303,164,0.895319,0.317130,0.835366,0.459732,0.292624
1,label_2640,182,0.862484,0.277264,0.824176,0.414938,0.349134
4,label_1367,129,0.894018,0.261501,0.837209,0.398524,0.300475
11,label_3436,84,0.870286,0.136259,0.702381,0.228240,0.350486
8,label_764,106,0.864434,0.139211,0.566038,0.223464,0.386319
7,label_3731,85,0.888492,0.133523,0.552941,0.215103,0.405895


Average validation metrics across trained labels:
accuracy     0.893097
precision    0.290815
recall       0.749137
f1           0.397086
log_loss     0.319021
dtype: float64
